# Notebook 2: Pipeline NASA POWER (2020-2025)

**Objetivo**: descargar datos climáticos mensuales para los 14 distritos representativos seleccionados, limpiarlos y exportar dataset climático en formato largo.

**Decisiones clave**:
- 14 distritos elegidos por relevancia agrícola y diversidad ecológica (costa, sierra, selva, yunga, altiplano)
- 10 variables crudas (sin índices derivados — esos se calculan en notebook 3 si se quieren)
- Sentinel `-999` de NASA → NaN
- Se descarta el resumen anual (`MM=13`) que NASA agrega automáticamente
- Output: formato largo (`distrito`, `region`, `anio`, `mes_num`, `<variables climáticas>`)
- Manejo robusto de errores: si una llamada falla, se reporta y reintenta

In [12]:
import pandas as pd
import numpy as np
import requests
import time
import unicodedata
from pathlib import Path

pd.set_option('display.max_columns', 20)

## 1. Configuración

**Coordenadas finales** acordadas tras revisión geográfica. La columna `region` permite asociar cada distrito a su producción correspondiente en el merge del notebook 3.

In [13]:
# >>>>>>>>>>> AJUSTAR ESTA RUTA <<<<<<<<<<<
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent

RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_BASE = ROOT / "Datasets Originales"
RUTA_OUTPUT.mkdir(parents=True, exist_ok=True)

# Endpoint NASA POWER (mensual, punto)
URL_NASA = "https://power.larc.nasa.gov/api/temporal/monthly/point"

# Rango temporal
ANIO_INICIO = 2020
ANIO_FIN = 2025

# Variables climaticas crudas (10)
PARAMETROS = [
    'T2M',              # Temperatura promedio
    'T2M_MAX',          # Temperatura maxima
    'T2M_MIN',          # Temperatura minima
    'PRECTOTCORR',      # Precipitacion corregida
    'RH2M',             # Humedad relativa a 2m
    'ALLSKY_SFC_SW_DWN',# Radiacion solar
    'WS2M',             # Velocidad viento a 2m
    'PS',               # Presion superficial
    'GWETROOT',         # Humedad zona radicular
    'TS',               # Temperatura superficial del suelo
    'T2MDEW',           # Punto de rocio
    'QV2M',             # Humedad especifica
]

# 14 distritos seleccionados (region sin tildes para coincidir con MIDAGRI)
DISTRITOS = [
    # Region          Distrito           Lat       Lon       Piso ecologico
    {'region':'Ica',         'distrito':'Chincha Alta','lat':-13.4099,'lon':-76.1324,'piso':'costa'},
    {'region':'La Libertad', 'distrito':'Viru',       'lat':-8.4143, 'lon':-78.7524,'piso':'costa'},
    {'region':'La Libertad', 'distrito':'Huamachuco', 'lat':-7.8121, 'lon':-78.0487,'piso':'sierra'},
    {'region':'La Libertad', 'distrito':'Cascas',     'lat':-7.4797, 'lon':-78.8178,'piso':'yunga'},
    {'region':'Piura',       'distrito':'Tambogrande','lat':-4.9269, 'lon':-80.3447,'piso':'bosque_seco'},
    {'region':'Piura',       'distrito':'Sullana',    'lat':-4.9039, 'lon':-80.6853,'piso':'valle_chira'},
    {'region':'Piura',       'distrito':'Canchaque',  'lat':-5.3760, 'lon':-79.6098,'piso':'sierra'},
    {'region':'San Martin',  'distrito':'Moyobamba',  'lat':-6.0344, 'lon':-76.9742,'piso':'selva_alto_mayo'},
    {'region':'San Martin',  'distrito':'Tocache',    'lat':-8.1877, 'lon':-76.5205,'piso':'selva_huallaga'},
    {'region':'Junin',       'distrito':'Perene',     'lat':-10.9489,'lon':-75.2264,'piso':'selva_alta'},
    {'region':'Junin',       'distrito':'Rio Tambo',  'lat':-11.1656,'lon':-74.2353,'piso':'selva_baja'},
    {'region':'Junin',       'distrito':'El Tambo',   'lat':-12.0313,'lon':-75.2222,'piso':'sierra'},
    {'region':'Puno',        'distrito':'Ilave',      'lat':-16.0866,'lon':-69.6354,'piso':'altiplano_lacustre'},
    {'region':'Puno',        'distrito':'Ayaviri',    'lat':-14.8864,'lon':-70.5889,'piso':'puna_alta'},
]

print(f"Distritos configurados: {len(DISTRITOS)}")
print(f"Variables a descargar: {len(PARAMETROS)}")
print(f"Periodo: {ANIO_INICIO}-{ANIO_FIN}")
print(f"Llamadas a NASA POWER esperadas: {len(DISTRITOS)}")
print(f"Filas esperadas en output: {len(DISTRITOS)} x {(ANIO_FIN-ANIO_INICIO+1)*12} meses = {len(DISTRITOS)*(ANIO_FIN-ANIO_INICIO+1)*12}")

Distritos configurados: 14
Variables a descargar: 12
Periodo: 2020-2025
Llamadas a NASA POWER esperadas: 14
Filas esperadas en output: 14 x 72 meses = 1008


## 2. Función de descarga con manejo de errores

A diferencia del script original (que ignoraba silenciosamente los fallos), esta versión:
- Reintenta hasta 3 veces si la API falla
- Lanza error si tras los reintentos no obtiene respuesta válida
- Reporta el progreso de cada llamada

In [14]:
def descargar_nasa(distrito_info, parametros, anio_ini, anio_fin, max_reintentos=3):
    """
    Descarga datos NASA POWER para un punto. 
    Devuelve DataFrame con index YYYYMM y columnas = parametros.
    """
    params = {
        "start": str(anio_ini),
        "end": str(anio_fin),
        "latitude": distrito_info["lat"],
        "longitude": distrito_info["lon"],
        "community": "AG",
        "parameters": ",".join(parametros),
        "format": "JSON"
    }
    
    for intento in range(1, max_reintentos + 1):
        try:
            r = requests.get(URL_NASA, params=params, timeout=60)
            if r.status_code == 200:
                data = r.json()
                if "properties" not in data or "parameter" not in data["properties"]:
                    raise ValueError(f"Respuesta sin estructura esperada para {distrito_info['distrito']}")
                df = pd.DataFrame(data["properties"]["parameter"])
                return df
            else:
                print(f"  [intento {intento}/{max_reintentos}] HTTP {r.status_code} para {distrito_info['distrito']}")
        except Exception as e:
            print(f"  [intento {intento}/{max_reintentos}] Error: {e}")
        
        if intento < max_reintentos:
            time.sleep(2 ** intento)  # backoff exponencial: 2s, 4s, 8s...
    
    raise RuntimeError(f"Fallo definitivo descargando {distrito_info['distrito']} tras {max_reintentos} intentos")

## 3. Descarga para los 14 distritos

Llamada secuencial (NASA POWER tiene rate limits, no recomiendo paralelizar). El total son ~14 requests, no toma más de 1-2 minutos.

In [15]:
datos_distritos = []
print("Iniciando descarga de NASA POWER...")
print("="*60)

for i, dist in enumerate(DISTRITOS, 1):
    print(f"[{i}/{len(DISTRITOS)}] {dist['region']:>12} - {dist['distrito']:<15} ({dist['lat']:>8.4f}, {dist['lon']:>9.4f})")
    df_raw = descargar_nasa(dist, PARAMETROS, ANIO_INICIO, ANIO_FIN)
    df_raw = df_raw.reset_index().rename(columns={"index": "fecha_yyyymm"})
    df_raw["distrito"] = dist["distrito"]
    df_raw["region"] = dist["region"]
    df_raw["piso"] = dist["piso"]
    df_raw["lat"] = dist["lat"]
    df_raw["lon"] = dist["lon"]
    datos_distritos.append(df_raw)

df_nasa = pd.concat(datos_distritos, ignore_index=True)
print(f"\n✓ Descarga completa. Shape: {df_nasa.shape}")

Iniciando descarga de NASA POWER...
[1/14]          Ica - Chincha Alta    (-13.4099,  -76.1324)
[2/14]  La Libertad - Viru            ( -8.4143,  -78.7524)
[3/14]  La Libertad - Huamachuco      ( -7.8121,  -78.0487)
[4/14]  La Libertad - Cascas          ( -7.4797,  -78.8178)
[5/14]        Piura - Tambogrande     ( -4.9269,  -80.3447)
[6/14]        Piura - Sullana         ( -4.9039,  -80.6853)
[7/14]        Piura - Canchaque       ( -5.3760,  -79.6098)
[8/14]   San Martin - Moyobamba       ( -6.0344,  -76.9742)
[9/14]   San Martin - Tocache         ( -8.1877,  -76.5205)
[10/14]        Junin - Perene          (-10.9489,  -75.2264)
[11/14]        Junin - Rio Tambo       (-11.1656,  -74.2353)
[12/14]        Junin - El Tambo        (-12.0313,  -75.2222)
[13/14]         Puno - Ilave           (-16.0866,  -69.6354)
[14/14]         Puno - Ayaviri         (-14.8864,  -70.5889)

✓ Descarga completa. Shape: (1092, 18)


## 4. Limpieza

Tres pasos en orden:
1. **Eliminar resumen anual**: NASA inserta una fila con `MM=13` que es el promedio anual. La quitamos porque ya tenemos los 12 meses.
2. **Reemplazar sentinel `-999`**: NASA usa este valor cuando un parámetro no está disponible para un punto (por ejemplo, GWETROOT a veces no aplica en zonas costeras o de alta montaña).
3. **Parsear fecha**: separar `'202001'` en `anio=2020, mes_num=1`.

In [16]:
# 1. Eliminar resumen anual (MM=13)
n_antes = len(df_nasa)
df_nasa = df_nasa[~df_nasa["fecha_yyyymm"].astype(str).str.endswith("13")].copy()
print(f"Filas eliminadas (resumen anual MM=13): {n_antes - len(df_nasa)}")

# 2. Reemplazar -999 por NaN
n_neg999 = (df_nasa[PARAMETROS] == -999).sum().sum()
df_nasa[PARAMETROS] = df_nasa[PARAMETROS].replace(-999, np.nan)
print(f"Valores -999 convertidos a NaN: {n_neg999}")

# 3. Parsear fecha
df_nasa["anio"] = df_nasa["fecha_yyyymm"].astype(str).str[:4].astype(int)
df_nasa["mes_num"] = df_nasa["fecha_yyyymm"].astype(str).str[4:6].astype(int)

# Reordenar columnas
cols_meta = ["distrito", "region", "piso", "lat", "lon", "anio", "mes_num"]
df_nasa = df_nasa[cols_meta + PARAMETROS]
df_nasa = df_nasa.sort_values(["region", "distrito", "anio", "mes_num"]).reset_index(drop=True)

# Lowercase nombres de variables (consistencia con MIDAGRI)
df_nasa.columns = [c.lower() for c in df_nasa.columns]

print(f"\n✓ Limpieza completa.")
print(f"Shape final: {df_nasa.shape}")
print(f"Esperado: {len(DISTRITOS)} distritos × {(ANIO_FIN-ANIO_INICIO+1)*12} meses = {len(DISTRITOS)*(ANIO_FIN-ANIO_INICIO+1)*12}")

Filas eliminadas (resumen anual MM=13): 84
Valores -999 convertidos a NaN: 14

✓ Limpieza completa.
Shape final: (1008, 19)
Esperado: 14 distritos × 72 meses = 1008


## 5. Validación

Verificar que:
- Cada distrito tiene exactamente 72 filas (6 años × 12 meses)
- Hay datos para todo el rango temporal
- Los rangos de cada variable son razonables

In [17]:
print("=== VALIDACIÓN POR DISTRITO ===\n")
print(f"{'Region':<13} {'Distrito':<15} {'Filas':>6} {'Anios cubiertos':<18}")
print("-"*60)
for (region, distrito), grupo in df_nasa.groupby(['region','distrito']):
    anios = sorted(grupo['anio'].unique())
    n_filas = len(grupo)
    print(f"{region:<13} {distrito:<15} {n_filas:>6} {min(anios)}-{max(anios)}")

print("\n=== NaN por variable ===")
nan_pct = df_nasa[[c.lower() for c in PARAMETROS]].isna().mean() * 100
for var, pct in nan_pct.items():
    if pct > 0:
        print(f"  {var}: {pct:.1f}% NaN")
if nan_pct.sum() == 0:
    print("  Sin NaN en ninguna variable")

print("\n=== Estadísticas (sanity check) ===")
print(df_nasa[[c.lower() for c in PARAMETROS]].describe().round(2).T[['min','mean','max']])

=== VALIDACIÓN POR DISTRITO ===

Region        Distrito         Filas Anios cubiertos   
------------------------------------------------------------
Ica           Chincha Alta        72 2020-2025
Junin         El Tambo            72 2020-2025
Junin         Perene              72 2020-2025
Junin         Rio Tambo           72 2020-2025
La Libertad   Cascas              72 2020-2025
La Libertad   Huamachuco          72 2020-2025
La Libertad   Viru                72 2020-2025
Piura         Canchaque           72 2020-2025
Piura         Sullana             72 2020-2025
Piura         Tambogrande         72 2020-2025
Puno          Ayaviri             72 2020-2025
Puno          Ilave               72 2020-2025
San Martin    Moyobamba           72 2020-2025
San Martin    Tocache             72 2020-2025

=== NaN por variable ===
  allsky_sfc_sw_dwn: 1.4% NaN

=== Estadísticas (sanity check) ===
                     min   mean    max
t2m                 3.34  18.45  29.19
t2m_max            13

## 6. Tratamiento de NaN (si los hay)

Si alguna variable tiene NaN, los interpolamos por distrito (cada distrito tiene 72 puntos temporales consecutivos, así que una interpolación lineal preserva la estructura). Esto solo aplica a casos raros como GWETROOT en zonas donde el modelo de NASA no aplica.

* Ejemplo de interpolación lineal para un distrito

In [33]:
ejemplo_col = "allsky_sfc_sw_dwn"   # radiación solar
ejemplo_distrito = "Ilave"

serie_original = df_nasa[df_nasa["distrito"] == ejemplo_distrito][ejemplo_col].copy()
serie_interp = serie_original.interpolate(method="linear").ffill().bfill()

nan_indices = serie_original[serie_original.isna()].index

if len(nan_indices) > 0:
    idx_nan = nan_indices[0]
    pos_nan = serie_original.index.get_loc(idx_nan)

    # Mostrar fragmento alrededor del NaN (2 filas antes y 2 después)
    start = max(pos_nan-2, 0)
    end = min(pos_nan+3, len(serie_original))

    fragmento = pd.DataFrame({
        "original": serie_original.iloc[start:end],
        "interpolado": serie_interp.iloc[start:end]
    })

    print("\n=== Fragmento con NaN y vecinos ===")
    print(fragmento)

    # Explicación según posición
    if 0 < pos_nan < len(serie_original)-1:
        valor_prev = serie_original.iloc[pos_nan-1]
        valor_next = serie_original.iloc[pos_nan+1]
        print(f"\nNaN reemplazado por {serie_interp.iloc[pos_nan]:.2f}, "
              f"calculado entre {valor_prev:.2f} y {valor_next:.2f}")
    elif pos_nan == 0:
        valor_next = serie_original.iloc[pos_nan+1]
        print(f"\nNaN al inicio → reemplazado por {serie_interp.iloc[pos_nan]:.2f}, "
              f"relleno hacia adelante desde {valor_next:.2f}")
    elif pos_nan == len(serie_original)-1:
        valor_prev = serie_original.iloc[pos_nan-1]
        print(f"\nNaN al final → reemplazado por {serie_interp.iloc[pos_nan]:.2f}, "
              f"relleno hacia atrás desde {valor_prev:.2f}")
else:
    print("No se encontraron NaN en esta serie para mostrar el ejemplo.")




=== Fragmento con NaN y vecinos ===
     original  interpolado
861     26.18        26.18
862     26.37        26.37
863       NaN        26.37

NaN al final → reemplazado por 26.37, relleno hacia atrás desde 26.37


In [34]:
cols_clima = [c.lower() for c in PARAMETROS]
n_nan_antes = df_nasa[cols_clima].isna().sum().sum()

if n_nan_antes > 0:
    print(f"Interpolando {n_nan_antes} valores NaN por distrito...")
    for col in cols_clima:
        df_nasa[col] = (df_nasa.groupby('distrito')[col]
                       .transform(lambda x: x.interpolate(method='linear').ffill().bfill()))
    n_nan_despues = df_nasa[cols_clima].isna().sum().sum()
    print(f"NaN tras interpolación: {n_nan_despues}")
    if n_nan_despues > 0:
        print(f"  (Si quedan NaN, son distritos donde la variable no aplica en absoluto)")
else:
    print("No hay NaN, no se requiere interpolación.")

Interpolando 14 valores NaN por distrito...
NaN tras interpolación: 0


## 7. Exportar

In [8]:
ruta = RUTA_OUTPUT / "nasa_2020_2025.csv"
df_nasa.to_csv(ruta, index=False, encoding='utf-8-sig')
print(f"Exportado: {ruta}")
print(f"Total filas: {len(df_nasa):,}")
print(f"\nMuestra:")
print(df_nasa.head(8).to_string(index=False))

print(f"\n=== LISTO ===")
print(f"Próximo paso: ejecutar notebook 03_merge_y_filtrado.ipynb")

Exportado: C:\Users\Joyssie\Documents\Data Minning\Proyecto\Outputs\nasa_2020_2025.csv
Total filas: 1,008

Muestra:
    distrito region  piso      lat      lon  anio  mes_num   t2m  t2m_max  t2m_min  prectotcorr  rh2m  allsky_sfc_sw_dwn  ws2m    ps  gwetroot    ts  t2mdew  qv2m
Chincha Alta    Ica costa -13.4099 -76.1324  2020        1 21.96    27.55    18.21         0.12 78.45              22.25  2.99 95.82      0.33 23.95   17.83 13.41
Chincha Alta    Ica costa -13.4099 -76.1324  2020        2 22.85    27.98    19.51         0.16 79.22              23.67  2.84 95.77      0.33 24.72   18.86 14.31
Chincha Alta    Ica costa -13.4099 -76.1324  2020        3 22.62    27.86    19.17         0.20 78.16              22.84  2.70 95.82      0.33 24.34   18.41 13.90
Chincha Alta    Ica costa -13.4099 -76.1324  2020        4 21.48    27.26    16.69         0.04 73.38              22.14  2.70 95.90      0.33 22.83   16.13 12.10
Chincha Alta    Ica costa -13.4099 -76.1324  2020        5 20.00    2